In [ ]:
import json
import yfinance as yf
from datetime import datetime, timedelta
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [ ]:
def get_stock_price(ticker: str, start_date: str, end_date: str) -> str:
    """
    Retrieve stock price data for a given ticker symbol within a date range.

    Args:
        ticker: Stock ticker symbol (e.g., 'AAPL', 'GOOGL', 'MSFT')
        start_date: Start date in 'YYYY-MM-DD' format
        end_date: End date in 'YYYY-MM-DD' format

    Returns:
        JSON string with stock price data
    """
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(start=start_date, end=end_date)

        if hist.empty:
            return json.dumps({
                "error": f"No data found for {ticker} between {start_date} and {end_date}"
            })

        # Format the data
        prices = []
        for date, row in hist.iterrows():
            prices.append({
                "date": date.strftime("%Y-%m-%d"),
                "open": round(row["Open"], 2),
                "high": round(row["High"], 2),
                "low": round(row["Low"], 2),
                "close": round(row["Close"], 2),
                "volume": int(row["Volume"])
            })

        result = {
            "ticker": ticker.upper(),
            "start_date": start_date,
            "end_date": end_date,
            "currency": stock.info.get("currency", "USD"),
            "company_name": stock.info.get("shortName", ticker),
            "number_of_trading_days": len(prices),
            "prices": prices
        }

        return json.dumps(result, indent=2)

    except Exception as e:
        return json.dumps({"error": str(e)})

In [ ]:
# Quick test of the function
print("Testing Yahoo Finance function...")
test_result = get_stock_price("AAPL", "2024-01-02", "2024-01-05")
print(test_result)

In [ ]:
# Define Tool Schema for the LLM
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "Retrieve historical stock price data for a given ticker symbol within a specified date range. Returns open, high, low, close prices and volume for each trading day.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {
                        "type": "string",
                        "description": "The stock ticker symbol, e.g., 'AAPL' for Apple, 'GOOGL' for Google, 'MSFT' for Microsoft"
                    },
                    "start_date": {
                        "type": "string",
                        "description": "The start date for the price query in 'YYYY-MM-DD' format"
                    },
                    "end_date": {
                        "type": "string",
                        "description": "The end date for the price query in 'YYYY-MM-DD' format"
                    }
                },
                "required": ["ticker", "start_date", "end_date"]
            }
        }
    }
]

# Map function names to actual callable functions
available_functions = {
    "get_stock_price": get_stock_price
}

In [ ]:
# tool/function-calling support via its chat template.
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading model: {MODEL_NAME}")
print("This may take a minute...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print(f"Model loaded successfully on {model.device}")

In [ ]:
# Define the Chat & Tool-Calling Pipeline
def generate_response(messages, max_new_tokens=512):
    """Generate a response from the model given a list of messages."""
    text = tokenizer.apply_chat_template(
        messages,
        tools=tools,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.1,
            top_p=0.9,
            repetition_penalty=1.1
        )

    # Decode only the newly generated tokens
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return response


def parse_tool_calls(response_text):
    """
    Parse tool calls from the model response.
    Qwen2.5-Instruct uses a specific format for tool calls.
    """
    tool_calls = []

    # Qwen2.5 format: function name on one line, then JSON arguments
    # Pattern: <tool_call>\n{"name": "...", "arguments": {...}}\n</tool_call>
    import re

    # Try to find tool_call tags
    tool_call_pattern = r'<tool_call>\s*(\{.*?\})\s*</tool_call>'
    matches = re.findall(tool_call_pattern, response_text, re.DOTALL)

    for match in matches:
        try:
            call_data = json.loads(match)
            tool_calls.append({
                "name": call_data.get("name"),
                "arguments": call_data.get("arguments", {})
            })
        except json.JSONDecodeError:
            continue

    # Fallback: try to find JSON with function name and arguments directly
    if not tool_calls:
        # Look for patterns like: get_stock_price\n{"ticker": ...}
        func_pattern = r'(get_stock_price)\s*\n\s*(\{.*?\})'
        matches = re.findall(func_pattern, response_text, re.DOTALL)
        for func_name, args_str in matches:
            try:
                args = json.loads(args_str)
                tool_calls.append({"name": func_name, "arguments": args})
            except json.JSONDecodeError:
                continue

    # Another fallback: look for any JSON that contains our expected keys
    if not tool_calls:
        json_pattern = r'\{[^{}]*"ticker"[^{}]*\}'
        matches = re.findall(json_pattern, response_text)
        for match in matches:
            try:
                args = json.loads(match)
                if "ticker" in args and "start_date" in args:
                    tool_calls.append({
                        "name": "get_stock_price",
                        "arguments": args
                    })
            except json.JSONDecodeError:
                continue

    return tool_calls


def chat_with_tools(user_message):
    """
    Full pipeline: user message → LLM → tool call → execute → LLM → final answer
    """
    print(f"\n{'='*70}")
    print(f"👤 User: {user_message}")
    print(f"{'='*70}")

    # Step 1: Send user message to the model with tool definitions
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful financial assistant. You have access to a tool "
                "that can retrieve historical stock prices. When a user asks about "
                "stock prices, use the get_stock_price tool. Always extract the "
                "ticker symbol and date range from the user's query."
            )
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    print("\n🤖 Step 1: Asking LLM to interpret the request...")
    response = generate_response(messages)
    print(f"   LLM raw response:\n   {response[:500]}")

    # Step 2: Check if the model wants to call a tool
    tool_calls = parse_tool_calls(response)

    if tool_calls:
        print(f"\n🔧 Step 2: LLM requested {len(tool_calls)} tool call(s)")

        for i, tool_call in enumerate(tool_calls):
            func_name = tool_call["name"]
            func_args = tool_call["arguments"]

            print(f"\n   Tool Call {i+1}:")
            print(f"   Function: {func_name}")
            print(f"   Arguments: {json.dumps(func_args, indent=4)}")

            # Step 3: Execute the function
            if func_name in available_functions:
                print(f"\n📊 Step 3: Executing {func_name}...")
                function_response = available_functions[func_name](**func_args)

                # Parse for display
                try:
                    parsed = json.loads(function_response)
                    if "error" in parsed:
                        print(f"   ❌ Error: {parsed['error']}")
                    else:
                        print(f"   ✅ Retrieved {parsed.get('number_of_trading_days', 0)} "
                              f"trading days for {parsed.get('company_name', func_args['ticker'])}")
                except:
                    pass

                # Step 4: Send the tool result back to the model
                messages.append({
                    "role": "assistant",
                    "content": response
                })
                messages.append({
                    "role": "tool",
                    "name": func_name,
                    "content": function_response
                })

        # Step 5: Get the final response from the model
        print(f"\n🤖 Step 4: Asking LLM to summarize the results...")
        final_response = generate_response(messages, max_new_tokens=1024)
        print(f"\n{'='*70}")
        print(f"🤖 Assistant: {final_response}")
        print(f"{'='*70}")

        return final_response

    else:
        # No tool call detected — return the direct response
        print(f"\n   (No tool call detected — returning direct response)")
        print(f"\n{'='*70}")
        print(f"🤖 Assistant: {response}")
        print(f"{'='*70}")

        return response


In [ ]:
# Example 1: Simple stock price query
chat_with_tools("Get the price for AAPL from 2024-12-01 to 2024-12-10")

In [ ]:
chat_with_tools("What was Microsoft's stock price between 2024-11-01 and 2024-11-15?")

In [ ]:
chat_with_tools("Show me Tesla stock prices from January 15 2025 to January 20 2025")

In [ ]:
#Interactive Chat Loop (Optional)
def interactive_chat():
    """Run an interactive chat session."""
    print("\n" + "="*70)
    print("💬 INTERACTIVE STOCK PRICE CHAT")
    print("="*70)
    print("Ask about stock prices! Examples:")
    print("  - 'Get the price for AAPL from 2024-01-01 to 2024-01-31'")
    print("  - 'What was Google stock price in December 2024?'")
    print("  - 'Show me NVDA prices from 2024-06-01 to 2024-06-30'")
    print("  - Type 'quit' to exit")
    print("="*70)

    while True:
        user_input = input("\n👤 You: ").strip()

        if user_input.lower() in ['quit', 'exit', 'q']:
            print("👋 Goodbye!")
            break

        if not user_input:
            continue

        chat_with_tools(user_input)

In [ ]:
interactive_chat()